# UC2 — Metal Surface (Magnetic Tile): 0. Setup

Welcome to the **Cosmos AnomalyGen** step-by-step tutorial for **UC2: Metal Surface (Magnetic Tile)**.

Cosmos AnomalyGen is a diffusion-based pipeline for **few-shot synthetic anomaly
data generation**. Starting from a handful of real defect examples, it fine-tunes
a small set of adapter modules on top of a frozen Cosmos-Predict2 Text-to-Image
model, then inpaints realistic defects onto clean images at controlled locations.

This use case covers magnetic-tile surface defects (the classic Magnetic-Tile-Defect dataset).

> ### Prerequisite — build the `cosmos-predict2` environment first
>
> These notebooks assume the **`cosmos-predict2` conda environment already
> exists**. It provides the full runtime (PyTorch + CUDA 12.8, flash-attn,
> Transformer Engine, Apex, `huggingface_hub`, …) that every step relies
> on. It is **not** created by this notebook.
>
> If you have not set it up yet, run the top-level environment-setup notebook
> **once** and then return here:
> **[tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb)**
> (§0.1 *Setting Up the Environment* — the compile-heavy install of flash-attn /
> Transformer Engine / Apex).

**The UC2 tutorial is split into seven notebooks — run 0→5 in order for the manual
walkthrough, or jump to 6 to run everything through a single agent prompt:**

| # | Notebook | What you do |
|---|---|---|
| 0 | **0-setup** (this notebook) | Check the environment, authenticate Hugging Face, download checkpoints |
| 1 | [1-dataset-preparation](./1-dataset-preparation.ipynb) | Fetch & organize the Metal Surface dataset |
| 2 | [2-training](./2-training.ipynb) | Fine-tune the AnomalyGen modules (or use the released checkpoint) |
| 3 | [3-auto-mask-placement](./3-auto-mask-placement.ipynb) | Build the generation testcase (defect-mask placement) |
| 4 | [4-generation](./4-generation.ipynb) | Generate synthetic anomaly images + evaluate |
| 5 | [5-pseudo-labeling](./5-pseudo-labeling.ipynb) | Produce COCO annotations & captions |
| 6 | [6-agentic-flow](./6-agentic-flow.ipynb) | Run the whole pipeline (2→5) from one Claude Code prompt |

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 0.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 0.1 Verify the GPU environment

Confirm the `cosmos-predict2` environment is importable and CUDA is available.

In [ ]:
!conda run -n cosmos-predict2 python -c "import torch; print('torch', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('gpus:', torch.cuda.device_count())"

## 0.2 Authenticate with Hugging Face

The checkpoints and (for UC2) the dataset are pulled from Hugging Face. Authenticate
**once** — either log in interactively (persists to `~/.cache/huggingface`) or export a token:

```bash
# interactive (recommended)
conda run -n cosmos-predict2 hf auth login
# or, one-shot for this session
export HF_TOKEN=<your-token>
```

Your token needs read access to the `nvidia/Cosmos-AnomalyGen-*` repos. Verify:

In [ ]:
!conda run -n cosmos-predict2 hf auth whoami

## 0.3 Download the base checkpoints

The pipeline builds on several frozen base models: Cosmos-Predict2-2B-Text2Image
(the diffusion backbone), a T5 text encoder, NV-DINOv2 (mask encoder), DINOv2-large
(correspondence backbone for evaluation), C-RADIOv3-B (FID backbone), SAM2 (mask
refinement) and Qwen3-VL (captioner). Download them with the repo's helper:

```bash
conda run -n cosmos-predict2 hf auth login   # if not already logged in
python -m scripts.download_checkpoints --model_types text2image --model_sizes 2B
```

> The base checkpoints total ~70 GB. If they are already present under
> `checkpoints/` (e.g. shared on this machine) you can skip this download.

## 0.4 Download the released UC2 AnomalyGen checkpoint

Each use case has a published, finetuned 2B AnomalyGen checkpoint (the small adapter
weights + its `ag_config.yaml`). Downloading it lets you **skip training** and jump
straight to generation. The helper stages it under `checkpoints/nvidia/`:

In [ ]:
!conda run -n cosmos-predict2 bash scripts/utilities/download_anomalygen_checkpoints.sh --uc metal --checkpoint-dir checkpoints

The generation script expects the weights at
`checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B/checkpoints/model/iter_000010000.pt`. The published repo ships the
`.pt` at the top level, so link it into the expected layout once:

> **Confirm the downloaded filename first.** List what the download produced
> (`ls checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B/`). If the `.pt` is named
> differently than `iter_000010000.pt` (e.g. `iter_10000.pt`), update the source
> name in the next cell so the symlink doesn't dangle.

In [ ]:
!mkdir -p checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B/checkpoints/model && \
 ln -sfn ../../iter_000010000.pt checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B/checkpoints/model/iter_000010000.pt && \
 conda run -n cosmos-predict2 python scripts/utilities/validate_checkpoint.py checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B --step 10000

`validate_checkpoint.py` prints the anomaly types the checkpoint supports —
these are the defects you can generate.

## Next Step

Proceed to [1-dataset-preparation.ipynb](./1-dataset-preparation.ipynb).